In [1]:
from pathlib import Path
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.optim as optim

import numpy as np
from tqdm import tqdm

In [2]:
rows, cols = 64, 64

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_DIR = Path("../data/dataset2-master/dataset2-master/images")

train_dir = DATA_DIR / "TRAIN"
test_dir = DATA_DIR / "TEST"

In [3]:
transform = transforms.Compose([
    transforms.Resize((rows, cols)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train = datasets.ImageFolder(train_dir, transform=transform)
test = datasets.ImageFolder(test_dir, transform=transform)

In [4]:
BATCH_SIZE = 32
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
class LeNet(nn.Module):
    def __init__(self, num_classes: int = 10):
        super(LeNet, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            
            nn.Flatten(1, -1),
            
            nn.Linear(16 * 13 * 13, 120),
            nn.Tanh(),
            
            nn.Linear(120, 84),
            nn.Tanh(),
            
            nn.Linear(84, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)
        

In [6]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=len(train_loader), desc="Training"):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * data.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model_original = LeNet(num_classes=4)
model_original.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_original.parameters(), lr=0.1, momentum=0.0, weight_decay=0.0)

cuda


In [8]:
train(model_original, device, train_loader, optimizer, criterion)
torch.save(model_original.state_dict(), "../models/lenet_original.pth")

Training: 100%|██████████| 312/312 [00:33<00:00,  9.37it/s]
